In [1]:
import gtsam
import numpy as np
from typing import List

from gtsam import Pose3, StateData
from gtsam.symbol_shorthand import X, V, L, Y, W
import time
import os
import argparse

from collections import defaultdict

In [2]:
TOL_NS = 1_000_000        # 1 ms timestamp matching tolerance
FRAME_STRIDE = 4          # use every 4th non-KF frame to start
MIN_OBS = 3               # min observations per landmark
MAX_REPROJ_PX = 5.0       # init outlier threshold
STATE_DT = 0.5  # GP state spacing [s]
FRAME_DT = 0.05           # EuRoC camera period (20 Hz)
LM_PRIOR_SIGMA = 0.1

OUT_DIR = "results_incremental/MH01"

# EuRoC cam0 intrinsics. ORB-SLAM3's EuRoC yaml resizes images 752x480 -> 600x350
# (Camera.newWidth/newHeight), so the pixels in the dumps live in the resized image.
# Pixels are already undistorted (mvKeysUn), so no distortion model here.
SX, SY = 600 / 752, 350 / 480
K = gtsam.Cal3_S2(458.654 * SX, 457.296 * SY, 0.0, 367.215 * SX, 248.375 * SY)

noise = gtsam.noiseModel.Robust.Create(
    gtsam.noiseModel.mEstimator.Huber.Create(1.345), gtsam.noiseModel.Isotropic.Sigma(2, 1.0))
noise_interp = gtsam.noiseModel.Isotropic.Sigma(2, 1.0)   # Huber inside the interp factor segfaults
noise_lm = None if LM_PRIOR_SIGMA is None else gtsam.noiseModel.Isotropic.Sigma(3, LM_PRIOR_SIGMA)

In [3]:
def s_to_ns(s):            # "1403636579.963555574"
    a, b = s.split('.')
    return int(a) * 10**9 + int(b.ljust(9, '0')[:9])

def ns_to_s(ns):           # 1403636579963555574 -> "1403636579.963555574"
    ns = int(ns)
    sign = '-' if ns < 0 else ''
    a, b = divmod(abs(ns), 10**9)
    return f"{sign}{a}.{b:09d}"

def nearest(ts_sorted, q):
    i = np.searchsorted(ts_sorted, q)
    cand = [j for j in (i - 1, i) if 0 <= j < len(ts_sorted)]
    j = min(cand, key=lambda j: abs(int(ts_sorted[j]) - q))
    return j, abs(int(ts_sorted[j]) - q)

In [4]:
def load_f_dataset(path):
    """Parse a TUM file with lines: t_ns tx ty tz qx qy qz qw -> (int64 ns array, [Pose3])."""
    times_ns, poses = [], []
    with open(path) as f:
        for line in f:
            tok = line.split()
            if not tok or tok[0].startswith('#'):
                continue
            times_ns.append(int(tok[0].split('.')[0]))
            tx, ty, tz, qx, qy, qz, qw = map(float, tok[1:8])
            poses.append(Pose3(gtsam.Rot3.Quaternion(qw, qx, qy, qz), gtsam.Point3(tx, ty, tz)))
    return np.array(times_ns, dtype=np.int64), poses

# ---------- load landmarks + keyframe observations ----------
def build_landmark_lookup_dict(path): #, first_frame_ts):
    """ Return dict of Landmark_id and its 3D world Pose"""
    P = {}
    first_frame_obs = []
    for line in open(path):
        s = line.split()
        if s[0] == "P":
            P[int(s[1])] = np.array(list(map(float, s[2:5])))
        elif s[0] == "O" and s[2] == "1403636579.913555622": # TODO: Make this generic for all dataset
            first_frame_obs.append([int(s[1]), float(s[3]), float(s[4])])
    return P, first_frame_obs

def load_per_frame_observations(path, f_times_ns):
    frame_obs = {}
    with open(path) as f:
        for line in f:
            s = line.split()
            temp = [int(s[0]), float(s[2]), float(s[3])]
            f_times_idx, dt = nearest(f_times_ns, s_to_ns(s[1]))
            if dt <= TOL_NS:
                if(f_times_ns[f_times_idx] in frame_obs.keys()):
                    frame_obs[f_times_ns[f_times_idx]].append(temp)
                else:
                    frame_obs[f_times_ns[f_times_idx]] = [temp]
    return frame_obs

## 1. Load and filter (unchanged)

In [5]:
# kf_times_ns, kf_poses = load_kf_dataset("MH_01_data/kf_dataset-MH01_mono.txt")

# Loading all frames with timestamps and  
f_times_ns, f_poses = load_f_dataset("MH_01_data/f_dataset-MH01_mono.txt")
P, first_frame_obs = build_landmark_lookup_dict("MH_01_data/MapPoints_KF_UVobs_dataset-MH01_mono.txt")

# load frame observations
per_frame_obs = load_per_frame_observations("MH_01_data/Frame_obs_MH01_mono.txt", f_times_ns)
per_frame_obs[1403636579913555712] = first_frame_obs


In [6]:
len(f_times_ns)

3679

In [7]:
len(per_frame_obs.keys())

3679

In [8]:
params = gtsam.ISAM2Params(); params.setRelinearizeThreshold(0.1); params.relinearizeSkip = 1
isam = gtsam.ISAM2(params)

states: List[StateData] = []
full_graph = gtsam.NonlinearFactorGraph()         # copy of every factor, for the final query
landmarks_in_graph = set()
lm_buffer = defaultdict(list)                     # lid -> [(j, u, v)] until it has MIN_OBS
pending = []                                      # measurement frames since the last state
causal = []                                       # newest state's pose right after its update
new_f, new_v = gtsam.NonlinearFactorGraph(), gtsam.Values()
Qc = 0.1 * np.ones(6)

n_plain = n_interp = n_rejected = 0
t_start = time.time()

In [9]:
N = len(f_times_ns)
f_times = (f_times_ns - f_times_ns[0]) * 1e-9          # seconds since the first frame

# ---- GP states on a uniform 0.5 s time grid (not tied to frame indices) ----
n_states = int(np.ceil(f_times[-1] / STATE_DT)) + 1
state_t = STATE_DT * np.arange(n_states)               # 0, 0.5, 1.0, ... ; last one >= last frame
state_t_ns = f_times_ns[0] + np.round(state_t * 1e9).astype(np.int64)   # absolute, for saving

# a frame that lands on a state time (within TOL_NS) gets plain factors on that state
state_of_frame = {}
for i, t in enumerate(state_t):
    j = int(np.argmin(np.abs(f_times - t)))
    if abs(f_times[j] - t) <= TOL_NS * 1e-9:
        state_of_frame[j] = i

def orb_pose_and_vel_at(t):
    """ORB-SLAM3 pose at time t (slerp between the two frames around t) and the
    body velocity between those two frames (= backward difference from the previous frame)."""
    b = int(np.clip(np.searchsorted(f_times, t), 1, N - 1))    # f_times[b-1] < t <= f_times[b]
    a = b - 1
    alpha = float(np.clip((t - f_times[a]) / (f_times[b] - f_times[a]), 0.0, 1.0))
    T = f_poses[a].slerp(alpha, f_poses[b])                    # clamps to the last frame past the end
    w = gtsam.Pose3.Logmap(f_poses[a].between(f_poses[b])) / (f_times[b] - f_times[a])
    return T, w

def reproj_factor(lid, j, u, v):
    """Plain factor on X(i) if frame j lands on a state time, else interpolated between its two states."""
    if j in state_of_frame:
        return gtsam.GenericProjectionFactorCal3_S2(
            gtsam.Point2(u, v), noise, X(state_of_frame[j]), L(lid), K), True
    i0 = int(np.searchsorted(state_t, f_times[j], side="right")) - 1   # t_i0 < t_j < t_i0+1
    proj = gtsam.GenericProjectionFactorCal3_S2(gtsam.Point2(u, v), noise_interp, Y(j), L(lid), K)
    return gtsam.WnoaInterpFactorPose3(proj, {states[i0], states[i0 + 1]},
                                       {StateData(Y(j), W(j), f_times[j])}, Qc), False

def reproj_err(T_WC, p, u, v):
    cam = gtsam.PinholeCameraCal3_S2(T_WC, K)
    try:
        uv = cam.project(gtsam.Point3(*p))
    except RuntimeError:                                       # point behind the camera
        return np.inf
    return float(np.hypot(uv[0] - u, uv[1] - v))
    

def add(factor_and_kind):
    global n_plain, n_interp
    fct, is_plain = factor_and_kind
    new_f.add(fct)
    n_plain += is_plain
    n_interp += not is_plain

In [10]:
# state_t

In [16]:
lo = 0
for i, t_i in enumerate(state_t):

    hi = int(np.searchsorted(f_times, t_i, side="right"))       # frames with t <= t_i
    seg = [j for j in range(lo, hi) if j % FRAME_STRIDE == 0 or j in state_of_frame]

    # print(lo, hi, seg)
    lo = hi

    # ---- new GP state at time t_i ----
    T0, w = orb_pose_and_vel_at(t_i)
    states.append(StateData(X(i), V(i), t_i))
    new_v.insert(X(i), T0)
    new_v.insert(V(i), w)

    if i == 0:
        new_f.add(gtsam.PriorFactorPose3(X(0), T0, gtsam.noiseModel.Isotropic.Sigma(6, 1e-6)))
    else:
        new_f.add(gtsam.WnoaMotionFactorPose3(states[i - 1], states[i], Qc))

    if i == 1:                                         # monocular scale: translation of X(1)
        new_f.add(gtsam.PriorFactorPose3(X(1), T0, gtsam.noiseModel.Diagonal.Sigmas([1e3] * 3 + [1e-4] * 3)))
    
    # ---- observations of the frames in (t_{i-1}, t_i] ----
    for jj in seg:
        for lid, u, v in per_frame_obs.get(f_times_ns[jj], []):
            if lid not in P:
                continue                                  # culled / merged by ORB-SLAM3
            if reproj_err(f_poses[jj], P[lid], u, v) >= MAX_REPROJ_PX:
                n_rejected += 1                           # outlier w.r.t. ORB-SLAM3 pose + map
                continue
    
            if lid in landmarks_in_graph:                 # landmark already a variable
                # fct, _ = 
                add(reproj_factor(lid, jj, u, v))
                # new_f.add(fct)
                continue
    
            # landmark not in the graph yet: hold the observation until it has MIN_OBS views
            lm_buffer[lid].append((jj, u, v))
            if len(lm_buffer[lid]) >= MIN_OBS:
                new_v.insert(L(lid), gtsam.Point3(*P[lid]))
                if noise_lm is not None:                  # weak prior: keeps low-parallax depth solvable
                    new_f.add(gtsam.PriorFactorPoint3(L(lid), gtsam.Point3(*P[lid]), noise_lm))
                landmarks_in_graph.add(lid)
                for j_obs, u_obs, v_obs in lm_buffer.pop(lid):   # flush all buffered views
                    # fct, _ = 
                    add(reproj_factor(lid, j_obs, u_obs, v_obs))
                    # new_f.add(fct)
    
    if i == 0:
        continue                                      # V(0) is unconstrained until the first WNOA factor
    isam.update(new_f, new_v)
    for k in range(new_f.size()):
        full_graph.add(new_f.at(k))
    new_f, new_v = gtsam.NonlinearFactorGraph(), gtsam.Values()
    causal.append((state_t_ns[i], isam.calculateEstimatePose3(X(i))))

    if i % 50 == 0 or i == n_states - 1:
        print(f"state {i:4d}/{n_states - 1}  t = {t_i:6.1f} s  "
              f"landmarks {len(landmarks_in_graph):5d}  factors {full_graph.size():6d}  "
              f"{time.time() - t_start:6.1f} s", flush=True)

0 1 [0]
1 11 [4, 8, 10]
11 21 [12, 16, 20]
21 31 [24, 28, 30]
31 41 [32, 36, 40]
41 51 [44, 48, 50]
51 61 [52, 56, 60]
61 71 [64, 68, 70]
71 81 [72, 76, 80]
81 91 [84, 88, 90]
91 101 [92, 96, 100]
101 111 [104, 108, 110]
111 121 [112, 116, 120]
121 131 [124, 128, 130]
131 141 [132, 136, 140]
141 150 [144, 148]
150 161 [150, 152, 156, 160]
161 171 [164, 168, 170]
171 181 [172, 176, 180]
181 191 [184, 188, 190]
191 201 [192, 196, 200]
201 211 [204, 208, 210]
211 221 [212, 216, 220]
221 231 [224, 228, 230]
231 241 [232, 236, 240]
241 251 [244, 248, 250]
251 261 [252, 256, 260]
261 271 [264, 268, 270]
271 281 [272, 276, 280]
281 290 [284, 288]
290 300 [290, 292, 296]
300 310 [300, 304, 308]
310 321 [310, 312, 316, 320]
321 331 [324, 328, 330]
331 341 [332, 336, 340]
341 351 [344, 348, 350]
351 361 [352, 356, 360]
361 371 [364, 368, 370]
371 381 [372, 376, 380]
381 391 [384, 388, 390]
391 401 [392, 396, 400]
401 411 [404, 408, 410]
411 421 [412, 416, 420]
421 431 [424, 428, 430]
431 441 [43

In [29]:
for _ in range(5):                                # a few extra relinearization passes at the end
    isam.update()
result = isam.calculateEstimate()
print(f"{n_states} states, {len(landmarks_in_graph)} landmarks, {n_plain} plain + "
      f"{n_interp} interpolated reprojection factors, {n_rejected} obs rejected (> {MAX_REPROJ_PX} px), "
      f"{len(lm_buffer)} landmarks never reached {MIN_OBS} obs | {time.time() - t_start:.1f} s")
print(f"Final error (full graph): {full_graph.error(result):.1f}")

369 states, 0 landmarks, 0 plain + 0 interpolated reprojection factors, 0 obs rejected (> 5.0 px), 0 landmarks never reached 3 obs | 1852.4 s
Final error (full graph): 0.0


## save: smoothed states, smoothed pose at every frame, causal (filter) states 

In [30]:
query_js = [j for j in range(N) if j not in state_of_frame]
interp_vals = gtsam.updateInterpValuesPose3(
    full_graph, result, set(states), {StateData(Y(j), W(j), f_times[j]) for j in query_js}, Qc)

def pose_at(j):
    return result.atPose3(X(state_of_frame[j])) if j in state_of_frame else interp_vals.atPose3(Y(j))


def save_tum(path, times_ns, poses):
    with open(path, "w") as f:
        for t, T in zip(times_ns, poses):
            p, q = T.translation(), T.rotation().toQuaternion()
            f.write(f"{int(t)} {p[0]:.9f} {p[1]:.9f} {p[2]:.9f} "
                    f"{q.x():.9f} {q.y():.9f} {q.z():.9f} {q.w():.9f}\n")
            
os.makedirs(OUT_DIR, exist_ok=True)
save_tum(f"{OUT_DIR}/gp_inc_states_MH01_mono.txt", state_t_ns,
         [result.atPose3(X(i)) for i in range(n_states)])
save_tum(f"{OUT_DIR}/gp_inc_full_MH01_mono.txt", f_times_ns, [pose_at(j) for j in range(N)])
save_tum(f"{OUT_DIR}/gp_inc_causal_states_MH01_mono.txt", [t for t, _ in causal],
         [T for _, T in causal])
print(f"Saved gp_inc_states / gp_inc_full / gp_inc_causal_states to {OUT_DIR}/")

NameError: name 'OUT_DIR' is not defined